In [0]:
from UTILS import environment, catalog_tools as ct

In [0]:
# =====================================================================================================================================
# For reading schemas of sets of parquet files in order to compare them
# Storage location
# ===================================================================================================================================
# Get environment and correct for prd / prod case
env = environment.currentEnv(spark)
if env.lower() == 'prd':
     env = 'prod'
else: 
    env = env.lower()

# Set Azure blob file system secure protocol, storage account, and container
protocol = "abfss://"
store = f"sa{env}bronzeingestion"
container = "landing"

# Use the storage account access key instead of SAS token
spark.conf.set(f"fs.azure.account.key.{store}.dfs.core.windows.net", dbutils.secrets.get(scope="bronzeingestion-secret-scope", key=f"sa-{env}-bronzeingestion-ak"))

# Build full path to files
# rootpath = "GoogleBigQuery/GoogleAnalytics/fnd-cloud-project/analytics_250303278"
rootpath = "mfcs/RDS_TSFHEAD_AUDIT"
rootpath = "mfcs/RDS_WV_TSFHEAD"
datapath = f"{protocol}{container}@{store}.dfs.core.windows.net/{rootpath}"

In [0]:
from collections import namedtuple

# A named tuple for capturing the schema summary of each dataset to consider processing
stat = namedtuple('stat', ['filepath', 'NumFields', 'NewFields'])

# Setup list to hold found schemas, test schema and difference
schemas = stats = []
base = comp = diff = None

# Get the list of directories
dirs = ct.directory_tree(dbutils, datapath)
# display(dirs)

for d in dirs:
    if d["level"] == 2:
        if base is None:
            base = spark.read.format("parquet").load(d["path"]).schema
            schemastring = ""
            for sf in base:
                schemastring += f"\n{sf.name: >30} : {str(sf.dataType)}"
            stats.append(stat(d["path"], len(base), schemastring))
        else:
            comp = spark.read.format("parquet").load(d["path"]).schema

        if type(base) != type(None) and type(comp) != type(None):
            diff = ct.schema_difference(base, comp)
            if len(diff) > 0:
                schemastring = ""
                for sf in diff:
                    schemastring += f"\n+++{sf[1].name: >30} : {str(sf[1].dataType)}"
                stats.append(stat(d["path"], len(comp), schemastring))

for s in stats:
    print(f"{s.filepath}\n{s.NumFields: >32} Fields\n{'-----------': >40}{s.NewFields}\n")

In [0]:
df_files = spark.read.format("parquet").load(d["path"])
df_files.createOrReplaceTempView("vw_files")
display(df_files)

In [0]:
from datetime import datetime

dt  = datetime.strptime('2025-09-25T07:33:12Z', '%Y-%m-%dT%H:%M:%S%z')

print(dt, type(dt))

df = spark.sql(f"select '{dt}' as dt, cast('{dt}' as timestamp) dt_ts")
df.createOrReplaceTempView('vw_dt')

dv = spark.sql("describe vw_dt")
display(dv)

In [0]:
help(ct)
